### Count frequency of SAR words in commit messages

In [12]:
import pandas as pd
import re
from collections import defaultdict, Counter
from bug_fix_list import bug_words
from internal_list import internal_words
from external_list import external_words
from functional_list import functional_words
from code_smell_list import smell_words

print("Starting memory-efficient SAR keyword frequency script...")

# Load dataset files from Hugging Face
print("Loading parquet files...")
all_pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
pr_commit_details_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet")
print("Parquet files loaded.")

# Merge metadata into one frame
print("Merging datasets...")
combined_df = pd.merge(
    all_pr_df[['id', 'agent']],
    pr_commit_details_df[['pr_id', 'message']],
    left_on='id',
    right_on='pr_id',
    how='left'
)

# Drop the redundant key column from pr_commit_details
combined_df = combined_df.drop(columns=['pr_id'])

# Remove rows where 'message' is NaN
combined_df = combined_df.dropna(subset=['message'])

print(combined_df)

print("Datasets merged.")

Starting memory-efficient SAR keyword frequency script...
Loading parquet files...
Parquet files loaded.
Merging datasets...
                 id        agent  \
26       3264933329  Claude_Code   
27       3264933329  Claude_Code   
28       3264933329  Claude_Code   
38       3265118634  Claude_Code   
39       3265118634  Claude_Code   
...             ...          ...   
1611128  2858527610        Devin   
1611129  2858527610        Devin   
1611130  2858527610        Devin   
1611131  2858527610        Devin   
1611132  2858527610        Devin   

                                                   message  
26       fix: Wait for all partitions in load_collectio...  
27       fix: Wait for all partitions in load_collectio...  
28       fix: Wait for all partitions in load_collectio...  
38       ファイルパス参照を相対パスに統一し、doc/からdocs/に統一\n\n- commands...  
39       ファイルパス参照を相対パスに統一し、doc/からdocs/に統一\n\n- commands...  
...                                                    ...  
1611128  Add do

In [16]:
# Combine all patterns
all_patterns = list(set(
    # bug_words +
    # internal_words +
    # external_words +
    # functional_words +
    smell_words
))

# Precompile regex patterns for each phrase (faster lookups)
compiled_patterns = {phrase: re.compile(re.escape(phrase), re.IGNORECASE) for phrase in all_patterns}

# Initialize counter structure
agent_phrase_counts = defaultdict(Counter)

print("Counting SAR phrase occurrences...")

# Process rows in chunks to save RAM
chunk_size = 50000  # tune for your RAM; 50k is safe for 16GB machines
for start in range(0, len(combined_df), chunk_size):
    end = start + chunk_size
    chunk = combined_df.iloc[start:end]

    for _, row in chunk.iterrows():
        agent = row["agent"]
        if pd.isna(agent):
            continue
        text_parts = [row.get("message", "")]
        text = " ".join(str(t) for t in text_parts if isinstance(t, str))
        if not text.strip():
            continue

        text_lower = text.lower()
        for phrase, pattern in compiled_patterns.items():
            # count occurrences of phrase in this text
            count = len(pattern.findall(text_lower))
            if count > 0:
                agent_phrase_counts[agent][phrase] += count

    print(f"  Processed rows {start:,}-{end:,}")

print("Counting complete. Converting to DataFrame...")

# Convert to table
frequency_df = pd.DataFrame(agent_phrase_counts).T.fillna(0).astype(int)
frequency_df.index.name = "Agent"

# Sum of all phrase counts per agent
frequency_df["Messages with SAR"] = frequency_df.sum(axis=1)

# Compute total number of commit messages per agent, independent of matches
print("Computing overall statistics...")
agent_total_messages = combined_df.groupby("agent")["message"].count().astype(int)
frequency_df["Total Messages"] = frequency_df.index.map(agent_total_messages)

# Calculate the percentage
frequency_df["% with SAR"] = (
    (frequency_df["Messages with SAR"] / frequency_df["Total Messages"]) * 100
).round(2)

# Save output
frequency_df.to_csv("phrase_frequency_.csv")
print("Saved table: phrase_frequency_.csv")

# Show preview
print(frequency_df.head())


Counting SAR phrase occurrences...
  Processed rows 0-50,000
  Processed rows 50,000-100,000
  Processed rows 100,000-150,000
  Processed rows 150,000-200,000
  Processed rows 200,000-250,000
  Processed rows 250,000-300,000
  Processed rows 300,000-350,000
  Processed rows 350,000-400,000
  Processed rows 400,000-450,000
  Processed rows 450,000-500,000
  Processed rows 500,000-550,000
  Processed rows 550,000-600,000
  Processed rows 600,000-650,000
  Processed rows 650,000-700,000
  Processed rows 700,000-750,000
Counting complete. Converting to DataFrame...
Computing overall statistics...
Saved table: phrase_frequency_.csv
             Avoid code duplication  Reduce code duplication  \
Agent                                                          
Claude_Code                       5                        2   
Copilot                           9                       41   
Devin                            14                       23   

             Remove code duplication  Elimin